# Análise exploratória — cenários de vibração

Este notebook responde às perguntas que moldam o detector, antes de ajustar qualquer modelo:

1. **O que temos?** Tamanho e duração de cada cenário, e o intervalo entre os períodos de treino e de predição.
2. **Como é a amostragem?** Intervalo entre leituras, buracos e duplicatas.
3. **Os dados estão limpos?** Valores ausentes, zeros, negativos e a fração do tempo com a máquina parada.
4. **Como é a máquina "parada"?** Se a flag `uptime` é confiável para remover as leituras com a máquina parada.
5. **Como os canais se distribuem?** Assimetria, efeito do logaritmo, correlação entre os eixos.
6. **O baseline se mantém?** Deslocamento do nível normal entre o período de treino e o de predição.
7. **O que os rótulos marcam?** Duração dos incidentes e o nível antes, durante e depois de cada um.
8. **Visualizador de cenários** — os seis canais de cada cenário, períodos de treino e de teste lado a lado.

Os dados são lidos com os mesmos loaders que o pipeline usa (`utils.data_loading`), então o que aparece aqui é exatamente o que o modelo recebe.


In [ ]:
import os
import sys
from pathlib import Path

# Roda a partir da raiz do repositório para que os caminhos relativos dos loaders funcionem.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.data_loading import load_all_scenarios
from utils.utils import timeseries_to_df

pd.set_option("display.max_rows", 100, "display.width", 200, "display.precision", 2)

VELOCITY = ["vel_rms_x", "vel_rms_y", "vel_rms_z"]
ACCELERATION = ["accel_rms_x", "accel_rms_y", "accel_rms_z"]
CHANNELS = VELOCITY + ACCELERATION
MAD_TO_SIGMA = 1.4826

# Uma cor por eixo, a mesma em todos os gráficos (paleta segura para daltonismo).
AXIS_COLORS = {"x": "#2a78d6", "y": "#eb6834", "z": "#1baf7a"}
FIT_COLOR, PRED_COLOR = "#2a78d6", "#eb6834"
INCIDENT_FILL = "rgba(208, 59, 59, 0.18)"
NEUTRAL = "#8a8985"
DIVERGING = [[0.0, "#2a78d6"], [0.5, "#f0efec"], [1.0, "#e34948"]]
# Marcas com números inteiros nos eixos log (o padrão do plotly rotula só o primeiro dígito das marcas menores).
LOG_TICKS = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]


def style(fig: go.Figure, title: str, height: int = 420, subtitle: str | None = None,
          legend: bool = True) -> go.Figure:
    # Visual comum: fundo branco, grade discreta. Título, subtítulo e legenda ficam empilhados em
    # faixas de pixels fixas acima da área do gráfico (coordenadas do container), então nunca se
    # sobrepõem entre si, aos títulos dos subgráficos ou aos dados, qualquer que seja a altura.
    title_px = 12
    legend_px = title_px + (52 if subtitle else 32)
    top_px = legend_px + (40 if legend else 8) + 16
    fig.update_layout(
        title=dict(
            text=title, x=0.01, xanchor="left", yref="container", y=1 - title_px / height, yanchor="top",
            font=dict(size=16), subtitle=dict(text=subtitle or "", font=dict(size=12, color="#52514e")),
        ),
        template="plotly_white",
        height=height,
        margin=dict(t=top_px, b=60, l=70, r=30),
        showlegend=legend,
        legend=dict(orientation="h", yref="container", y=1 - legend_px / height, yanchor="top",
                    x=0, xanchor="left", font=dict(size=12)),
        hovermode="x unified",
    )
    return fig

In [ ]:
scenarios = {}
for scenario_id, (fit_ts, pred_ts, incidents) in enumerate(load_all_scenarios(), start=1):
    scenarios[scenario_id] = {
        "fit": timeseries_to_df(fit_ts).set_index("sampled_at"),
        "pred": timeseries_to_df(pred_ts).set_index("sampled_at"),
        "incidents": incidents,
    }

print(f"{len(scenarios)} cenários carregados")
scenarios[1]["fit"].head()

## 1. Inventário

Uma linha por cenário: quanto dado existe para aprender o baseline, por quanto tempo o modelo precisa rodar e quão distantes estão os dois períodos.

In [ ]:
def inventory_row(scenario_id: int) -> dict:
    fit, pred = scenarios[scenario_id]["fit"], scenarios[scenario_id]["pred"]
    days = lambda df: (df.index[-1] - df.index[0]).total_seconds() / 86400
    return {
        "cenário": scenario_id,
        "linhas_treino": len(fit),
        "início_treino": fit.index[0].date(),
        "dias_treino": days(fit),
        "dias_intervalo": (pred.index[0] - fit.index[-1]).total_seconds() / 86400,
        "linhas_predição": len(pred),
        "início_predição": pred.index[0].date(),
        "dias_predição": days(pred),
        "incidentes": len(scenarios[scenario_id]["incidents"]),
    }


inventory = pd.DataFrame([inventory_row(i) for i in scenarios]).set_index("cenário")
display(inventory)
inventory[["dias_treino", "dias_intervalo", "dias_predição"]].describe().loc[["min", "50%", "max"]]

In [ ]:
fig = go.Figure()
for column, name, color in [
    ("dias_treino", "Período de treino", FIT_COLOR),
    ("dias_intervalo", "Intervalo (sem dados)", "#d8d7d2"),
    ("dias_predição", "Período de predição", PRED_COLOR),
]:
    fig.add_bar(
        y=inventory.index.astype(str), x=inventory[column], name=name, orientation="h",
        marker=dict(color=color, line=dict(color="white", width=2)),
        hovertemplate="%{x:.1f} dias<extra>" + name + "</extra>",
    )
style(fig, "Dados de treino, intervalo e dados de predição por cenário", height=760,
      subtitle="Duração de cada período em dias")
fig.update_layout(barmode="stack", hovermode="y unified", legend_traceorder="normal")
fig.update_yaxes(title_text="Cenário", autorange="reversed")
fig.update_xaxes(title_text="Dias")
fig.show()

## 2. Amostragem

O enunciado diz "aproximadamente a cada 10 minutos". O intervalo entre leituras consecutivas mostra quais cenários fogem disso e onde faltam dados por completo.

In [ ]:
def sampling_row(scenario_id: int, period: str) -> dict:
    index = scenarios[scenario_id][period].index
    minutes = index.to_series().diff().dt.total_seconds().div(60).dropna()
    return {
        "cenário": scenario_id,
        "período": period,
        "ordenado": index.is_monotonic_increasing,
        "timestamps_duplicados": int(index.duplicated().sum()),
        "intervalo_mediano_min": minutes.median(),
        "intervalo_p95_min": minutes.quantile(0.95),
        "intervalo_máx_min": minutes.max(),
        "buracos_acima_de_1h": int((minutes > 60).sum()),
    }


sampling = pd.DataFrame(
    [sampling_row(i, period) for i in scenarios for period in ("fit", "pred")]
).set_index(["cenário", "período"])

print("Séries fora de ordem:", int((~sampling["ordenado"]).sum()),
      "| timestamps duplicados:", int(sampling["timestamps_duplicados"].sum()))
print("\nCenários não amostrados a ~10 min:")
display(sampling[(sampling["intervalo_mediano_min"] - 10).abs() > 0.5])
print("Cenários com buracos de mais de 1 hora:")
display(sampling[sampling["buracos_acima_de_1h"] > 0])

In [ ]:
intervals = pd.concat(
    scenarios[i][period].index.to_series().diff().dt.total_seconds().div(60).dropna()
    for i in scenarios for period in ("fit", "pred")
)
bins = [0, 2, 5, 9, 11, 20, 30, 60, 120, 1e9]
labels = ["<2", "2-5", "5-9", "9-11", "11-20", "20-30", "30-60", "60-120", ">120"]
share = pd.cut(intervals, bins=bins, labels=labels).value_counts(normalize=True).reindex(labels)

fig = go.Figure(go.Bar(
    x=labels, y=share.values, marker_color=FIT_COLOR,
    text=[f"{v:.1%}" if v >= 0.0005 else f"{v:.2%}" for v in share.values], textposition="outside",
    cliponaxis=False,
    hovertemplate="%{x} min: %{y:.2%}<extra></extra>",
))
style(fig, "Intervalo entre leituras consecutivas", subtitle="Fração de todas as leituras, todos os cenários e períodos",
      legend=False)
fig.update_xaxes(title_text="Intervalo entre leituras consecutivas (minutos)")
fig.update_yaxes(title_text="Fração das leituras", tickformat=".0%", range=[0, share.max() * 1.15])
fig.show()

## 3. Qualidade dos dados

Valores ausentes, valores impossíveis (uma amplitude RMS não pode ser negativa) e quanto de cada período a máquina passa parada.

In [ ]:
def quality_row(scenario_id: int, period: str) -> dict:
    df = scenarios[scenario_id][period]
    return {
        "cenário": scenario_id,
        "período": period,
        "valores_ausentes": int(df[CHANNELS].isna().sum().sum()),
        "valores_negativos": int((df[CHANNELS] < 0).sum().sum()),
        "linhas_todas_zero": int((df[CHANNELS] == 0).all(axis=1).sum()),
        "fração_rodando": df["uptime"].mean(),
        "linhas_rodando": int(df["uptime"].sum()),
    }


quality = pd.DataFrame(
    [quality_row(i, period) for i in scenarios for period in ("fit", "pred")]
).set_index(["cenário", "período"])

print(quality[["valores_ausentes", "valores_negativos", "linhas_todas_zero"]].sum().to_string())
print("\nMenos linhas com a máquina rodando disponíveis para ajustar um baseline:")
display(quality.xs("fit", level="período").nsmallest(5, "linhas_rodando")[["fração_rodando", "linhas_rodando"]])

In [ ]:
running = quality["fração_rodando"].unstack("período")

fig = go.Figure()
for period, name, color in [("fit", "Período de treino", FIT_COLOR), ("pred", "Período de predição", PRED_COLOR)]:
    fig.add_bar(
        x=running.index.astype(str), y=running[period], name=name,
        marker=dict(color=color, line=dict(color="white", width=1)),
        hovertemplate="%{y:.0%}<extra>" + name + "</extra>",
    )
style(fig, "Fração das leituras feitas com a máquina rodando", height=440)
fig.update_layout(barmode="group", bargap=0.3)
fig.update_xaxes(title_text="Cenário")
fig.update_yaxes(title_text="Fração rodando", tickformat=".0%", range=[0, 1.05])
fig.show()

## 4. Rodando versus parada

As leituras com a máquina parada não podem chegar ao modelo: puxariam o baseline para baixo e inflariam sua dispersão. A pergunta é se o `uptime` separa os dois estados de forma limpa.

O gráfico compara o nível típico em cada estado, para a norma da velocidade e a norma da aceleração dos três eixos. A tabela depois conta as leituras do lado errado: marcadas como *rodando* mas no piso de ruído da máquina parada, e marcadas como *parada* mas em nível de operação.

In [ ]:
def norm(df: pd.DataFrame, columns: list[str]) -> pd.Series:
    return np.sqrt((df[columns] ** 2).sum(axis=1))


def state_row(scenario_id: int) -> dict:
    df = pd.concat([scenarios[scenario_id]["fit"], scenarios[scenario_id]["pred"]])
    running, stopped = df[df["uptime"]], df[~df["uptime"]]
    row = {"cenário": scenario_id}
    for name, columns in [("velocidade", VELOCITY), ("aceleração", ACCELERATION)]:
        row[f"{name}_rodando"] = norm(running, columns).median()
        row[f"{name}_parada"] = norm(stopped, columns).median() if len(stopped) else np.nan
    noise_floor = 3 * row["aceleração_parada"]
    row["rodando_no_piso_de_ruído"] = (norm(running, ACCELERATION) < noise_floor).mean()
    row["parada_em_nível_de_operação"] = (
        (norm(stopped, ACCELERATION) > 0.5 * row["aceleração_rodando"]).mean() if len(stopped) else np.nan
    )
    return row


states = pd.DataFrame([state_row(i) for i in scenarios]).set_index("cenário")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("Norma da velocidade (mm/s)", "Norma da aceleração (g)"))
for row, quantity in enumerate(["velocidade", "aceleração"], start=1):
    for state, name, color, symbol in [
        ("rodando", "Rodando (mediana)", FIT_COLOR, "circle"),
        ("parada", "Parada (mediana)", NEUTRAL, "diamond"),
    ]:
        fig.add_scatter(
            x=states.index.astype(str), y=states[f"{quantity}_{state}"], name=name, mode="markers",
            legendgroup=state, showlegend=row == 1,
            marker=dict(color=color, size=10, symbol=symbol, line=dict(color="white", width=2)),
            hovertemplate="%{y:.3g}<extra>" + name + "</extra>", row=row, col=1,
        )
    fig.update_yaxes(type="log", tickvals=LOG_TICKS, row=row, col=1)
style(fig, "Nível típico com a máquina rodando e parada", height=600,
      subtitle="Mediana da norma dos três eixos, por cenário (escala log)")
fig.update_annotations(font_size=13)
fig.update_xaxes(title_text="Cenário", row=2, col=1)
fig.show()

states["razão_velocidade"] = states["velocidade_rodando"] / states["velocidade_parada"]
states["razão_aceleração"] = states["aceleração_rodando"] / states["aceleração_parada"]
print("Piso de ruído com a máquina parada entre os cenários (mediana das leituras paradas):")
print(states[["velocidade_parada", "aceleração_parada"]].describe().loc[["min", "50%", "max"]].round(3).to_string())
print("\nCenários cuja velocidade rodando é menor que 3x a velocidade parada:")
display(states[states["razão_velocidade"] < 3][["velocidade_rodando", "velocidade_parada", "razão_velocidade", "razão_aceleração"]])
print("Leituras do lado errado da flag (norma da aceleração):")
display(states[["rodando_no_piso_de_ruído", "parada_em_nível_de_operação"]].describe().loc[["50%", "max"]])
display(states[states["rodando_no_piso_de_ruído"] > 0.05][["aceleração_rodando", "aceleração_parada", "rodando_no_piso_de_ruído"]])

### Transientes de partida

A primeira leitura depois de uma parada pode pegar a máquina ainda acelerando. Se essas leituras ficassem bem longe do nível estável, precisariam de tratamento especial.

In [ ]:
def startup_ratio(scenario_id: int) -> pd.Series:
    # Aceleração da primeira leitura rodando depois de uma parada, relativa à mediana com a máquina rodando.
    df = pd.concat([scenarios[scenario_id]["fit"], scenarios[scenario_id]["pred"]])
    level = norm(df, ACCELERATION)
    first_running = df["uptime"] & ~df["uptime"].shift(1, fill_value=True)
    return level[first_running] / level[df["uptime"]].median()


startup = pd.concat({i: startup_ratio(i) for i in scenarios}, names=["cenário"])
print(f"{len(startup)} partidas em todos os cenários")
print("Primeira leitura rodando relativa à mediana rodando do cenário (norma da aceleração):")
print(startup.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(2).to_string())
print(f"Partidas abaixo de metade do nível rodando: {(startup < 0.5).mean():.0%} | acima do dobro: {(startup > 2).mean():.0%}")

## 5. Distribuição dos canais (máquina rodando)

Duas propriedades decidem como os canais devem entrar num modelo multivariado:

- **Forma.** O PCA e o T² de Hotelling resumem os dados por médias e covariâncias, que só descrevem bem uma distribuição aproximadamente simétrica. Amplitudes RMS são positivas, então o reflexo comum é tirar o logaritmo; as células abaixo verificam se esse reflexo é correto para estes dados.
- **Correlação.** Se os seis canais se movem juntos, um modelo conjunto consegue distinguir "tudo subiu" de "um eixo subiu sozinho".

In [ ]:
def running_rows(scenario_id: int, period: str) -> pd.DataFrame:
    df = scenarios[scenario_id][period]
    return df.loc[df["uptime"], CHANNELS]


def tail_ratio(series: pd.Series) -> float:
    # Comprimento da cauda superior sobre o da cauda inferior (1 = simétrica, >1 = cauda superior longa).
    low, mid, high = series.quantile([0.01, 0.5, 0.99])
    return (high - mid) / (mid - low)


describe = pd.concat({i: running_rows(i, "fit").describe().T for i in scenarios}, names=["cenário", "canal"])
describe["cv"] = describe["std"] / describe["mean"]
print("Período de treino, só leituras rodando (três primeiros cenários):")
display(describe.loc[1:3])
print("Coeficiente de variação de um canal rodando, em todas as séries cenário-canal:")
print(describe["cv"].quantile([0.05, 0.5, 0.95]).round(2).to_string())

shape = pd.DataFrame({
    "assimetria_bruta": pd.concat({i: running_rows(i, "fit").skew() for i in scenarios}),
    "assimetria_log": pd.concat({i: np.log(running_rows(i, "fit")).skew() for i in scenarios}),
    "razão_caudas_bruta": pd.concat({i: running_rows(i, "fit").apply(tail_ratio) for i in scenarios}),
    "razão_caudas_log": pd.concat({i: np.log(running_rows(i, "fit")).apply(tail_ratio) for i in scenarios}),
})
print(f"\nForma das {len(shape)} séries cenário-canal, brutas e depois do logaritmo:")
display(shape.describe().loc[["25%", "50%", "75%"]])
print(f"Séries mais próximas da simetria depois do logaritmo: {(shape['assimetria_log'].abs() < shape['assimetria_bruta'].abs()).mean():.0%}")

In [ ]:
fig = go.Figure()
for column, name, color in [("assimetria_bruta", "Amplitude bruta", NEUTRAL), ("assimetria_log", "Amplitude em log", FIT_COLOR)]:
    fig.add_histogram(
        x=shape[column].clip(-8, 8), name=name, xbins=dict(start=-8, end=8, size=0.5),
        marker=dict(color=color, line=dict(color="white", width=1)), opacity=0.75,
    )
fig.add_vline(x=0, line=dict(color="#52514e", width=1, dash="dot"))
style(fig, "Assimetria de cada série cenário-canal", subtitle="Período de treino, máquina rodando")
fig.update_layout(barmode="overlay", hovermode="closest")
fig.update_xaxes(title_text="Assimetria (0 = simétrica; valores limitados a ±8)")
fig.update_yaxes(title_text="Número de séries")
fig.show()

In [ ]:
correlations = np.stack([np.log(running_rows(i, "fit")).corr().to_numpy() for i in scenarios])
median_correlation = pd.DataFrame(np.median(correlations, axis=0), index=CHANNELS, columns=CHANNELS)

fig = go.Figure(go.Heatmap(
    z=median_correlation.values, x=CHANNELS, y=CHANNELS, zmin=-1, zmax=1, colorscale=DIVERGING,
    text=median_correlation.round(2).values, texttemplate="%{text}", xgap=2, ygap=2,
    colorbar=dict(title="Correlação"), hovertemplate="%{y} vs %{x}: %{z:.2f}<extra></extra>",
))
style(fig, "Correlação mediana entre os canais em log nos cenários", height=520,
      subtitle="Período de treino, máquina rodando", legend=False)
fig.update_layout(hovermode="closest")
fig.update_yaxes(autorange="reversed")
fig.show()

upper = correlations[:, *np.triu_indices(len(CHANNELS), k=1)]
print(f"Correlação entre pares nos cenários: mediana {np.median(upper):.2f}, "
      f"percentis 5-95 de {np.quantile(upper, 0.05):.2f} a {np.quantile(upper, 0.95):.2f}")

## 6. O baseline se mantém do treino para a predição?

Um baseline aprendido no período de treino só é útil se a operação *normal* continuar parecida depois. Para cada canal, o gráfico mostra onde fica o período de predição fora dos incidentes rotulados, medido em desvios-padrão robustos do período de treino (mediana e MAD da amplitude em log, só leituras rodando). Valores perto de zero indicam que o baseline se manteve.

In [ ]:
def robust_baseline(scenario_id: int) -> tuple[pd.Series, pd.Series]:
    # Mediana e sigma baseado no MAD da amplitude em log nas leituras rodando do treino.
    log_fit = np.log(running_rows(scenario_id, "fit"))
    center = log_fit.median()
    return center, MAD_TO_SIGMA * (log_fit - center).abs().median()


def robust_z(scenario_id: int) -> pd.DataFrame:
    # Período de predição (leituras rodando) em sigmas robustos do período de treino.
    center, sigma = robust_baseline(scenario_id)
    return (np.log(running_rows(scenario_id, "pred")) - center) / sigma


def valid_incidents(scenario_id: int) -> list:
    return [inc for inc in scenarios[scenario_id]["incidents"] if inc.start < inc.end]


def inside_incident(index: pd.DatetimeIndex, scenario_id: int) -> np.ndarray:
    mask = np.zeros(len(index), dtype=bool)
    for incident in valid_incidents(scenario_id):
        mask |= (index >= incident.start) & (index <= incident.end)
    return mask


drift = pd.DataFrame({
    i: robust_z(i)[~inside_incident(robust_z(i).index, i)].median() for i in scenarios
}).T.rename_axis("cenário")

fig = go.Figure(go.Heatmap(
    z=drift.values, x=CHANNELS, y=[f"Cenário {i}" for i in drift.index], zmin=-6, zmax=6,
    colorscale=DIVERGING, text=drift.round(1).values, texttemplate="%{text}", textfont=dict(size=11),
    xgap=2, ygap=2, colorbar=dict(title="z robusto"),
    hovertemplate="%{y}, %{x}: %{z:.1f}<extra></extra>",
))
style(fig, "Deslocamento do nível normal do treino para a predição", height=860, legend=False,
      subtitle="Mediana do período de predição fora dos incidentes, em sigmas do treino (cor limitada a ±6)")
fig.update_layout(hovermode="closest")
fig.update_xaxes(side="top")
fig.update_yaxes(autorange="reversed")
fig.show()

shifted = drift.abs().max(axis=1)
print("Cenários cujo nível normal se deslocou mais de 2 sigma em algum canal:",
      {int(k): float(v) for k, v in shifted[shifted > 2].round(1).items()})

## 7. O que os rótulos marcam

Todo alerta é pontuado só contra as janelas rotuladas, então importa como essas janelas se relacionam com o sinal: quanto duram e se a vibração está de fato elevada *antes*, *durante* e *depois* delas.

Para cada incidente, a tabela dá a mediana do z robusto do canal mais desviado nas 24 h antes, durante e nas 24 h depois da janela.

In [ ]:
def incident_rows(scenario_id: int) -> list[dict]:
    z = robust_z(scenario_id)
    pred_end = scenarios[scenario_id]["pred"].index[-1]
    day = pd.Timedelta(hours=24)
    rows = []
    for incident in scenarios[scenario_id]["incidents"]:
        row = {
            "cenário": scenario_id,
            "início": incident.start,
            "horas": (incident.end - incident.start).total_seconds() / 3600,
        }
        if incident.start < incident.end:
            segments = {
                "antes": (incident.start - day, incident.start),
                "durante": (incident.start, incident.end),
                "depois": (incident.end, incident.end + day),
            }
            for name, (start, end) in segments.items():
                segment = z[(z.index >= start) & (z.index < end)]
                row[f"z_{name}"] = segment.median().max() if len(segment) else np.nan
            during = z[(z.index >= incident.start) & (z.index < incident.end)]
            row["canal_principal"] = during.median().idxmax() if len(during) else None
            row["leituras_rodando"] = len(during)
            row["horas_até_o_fim_dos_dados"] = (pred_end - incident.end).total_seconds() / 3600
        rows.append(row)
    return rows


incident_table = pd.DataFrame([row for i in scenarios for row in incident_rows(i)])
print(f"{len(incident_table)} incidentes rotulados em {incident_table['cenário'].nunique()} cenários; "
      f"{(inventory['incidentes'] == 0).sum()} cenários não têm nenhum.")
print("Incidentes cujo fim vem antes do início (nenhum alerta consegue encontrá-los):")
display(incident_table[incident_table["horas"] <= 0][["cenário", "início", "horas"]])
display(incident_table[incident_table["horas"] > 0].set_index(["cenário", "início"]).round(1))

In [ ]:
valid = incident_table[incident_table["horas"] > 0].reset_index(drop=True)
labels = [f"{row['cenário']} · {row['início']:%d/%m}" for _, row in valid.iterrows()]

fig = go.Figure()
for column, name, color, symbol in [
    ("z_antes", "24 h antes", NEUTRAL, "diamond"),
    ("z_durante", "Durante o incidente", "#d03b3b", "circle"),
    ("z_depois", "24 h depois", FIT_COLOR, "square"),
]:
    fig.add_scatter(
        x=labels, y=valid[column].clip(lower=0.1), name=name, mode="markers", offsetgroup=column,
        marker=dict(color=color, size=10, symbol=symbol, line=dict(color="white", width=2)),
        hovertemplate="%{y:.1f} sigma<extra>" + name + "</extra>",
    )
fig.add_hline(y=3, line=dict(color=NEUTRAL, width=1, dash="dot"))
fig.add_annotation(x=1, xref="paper", xanchor="left", y=np.log10(3), text=" 3σ", showarrow=False,
                   font=dict(size=11, color=NEUTRAL))
style(fig, "Nível do canal mais desviado em torno de cada incidente", height=520,
      subtitle="Mediana do z robusto em cada trecho (escala log, mínimo de 0,1)")
# Lado a lado dentro de cada incidente, para que marcadores com valores parecidos não se escondam.
fig.update_layout(margin_b=120, margin_r=50, hovermode="closest", scattermode="group", scattergap=0.4)
fig.update_xaxes(title_text="Cenário · início do incidente", tickangle=-60, title_standoff=12)
fig.update_yaxes(title_text="Mediana do z robusto", type="log", tickvals=LOG_TICKS)
fig.show()

weak = valid[valid["z_durante"] < 3]
print(f"Incidentes sem nenhum canal acima de 3 sigma (mediana) durante a janela: "
      f"{len(weak)} de {len(valid)} -> cenários {sorted(set(weak['cenário']))}")
early = valid[valid["z_antes"] >= 3]
print(f"Incidentes já acima de 3 sigma nas 24 h antes da janela: "
      f"{len(early)} -> cenários {sorted(set(early['cenário']))}")
late = valid[(valid["z_depois"] >= 3)]
print(f"Incidentes ainda acima de 3 sigma nas 24 h depois da janela: "
      f"{len(late)} -> cenários {sorted(set(late['cenário']))}")
print(f"Incidentes com menos de 4 h: {sorted(set(valid[valid['horas'] < 4]['cenário']))}")

## 8. Visualizador de cenários

Todos os cenários, com o período de treino (fit) à esquerda e o período de teste (predição) à direita. As duas colunas compartilham a escala vertical, então o nível dos dados de teste pode ser comparado diretamente com o baseline que o modelo aprende; o tempo entre os períodos (sem dados) fica de fora e aparece no subtítulo.

- **Duas linhas de cima** — RMS da velocidade e da aceleração, uma cor por eixo.
- **Faixa de baixo** — quando a máquina estava rodando.
- **Faixas vermelhas** — incidentes rotulados (todos caem no período de predição).

Com `hide_stopped=True` (padrão) as leituras feitas com a máquina parada ficam em branco, que é como o modelo vê os dados; passe `hide_stopped=False` para ver as séries brutas.


In [ ]:
def period_label(df: pd.DataFrame) -> str:
    return f"{df.index[0]:%d/%m} – {df.index[-1]:%d/%m/%Y}"


def axis_limit(values: pd.DataFrame) -> float:
    # Limite superior de um painel: um pico isolado (por exemplo, uma única leitura de 250 mm/s) achataria
    # todo o resto, então o eixo para no dobro do percentil 99,9 quando o máximo passa disso.
    stacked = values.stack()
    return float(min(stacked.max(), 2 * stacked.quantile(0.999)) * 1.05)


def plot_scenario(scenario_id: int = 1, hide_stopped: bool = True) -> None:
    scenario = scenarios[scenario_id]
    periods = [("fit", "Período de treino (fit)"), ("pred", "Período de teste (predição)")]
    days = {key: (scenario[key].index[-1] - scenario[key].index[0]).total_seconds() / 86400 for key, _ in periods}
    # A largura da coluna acompanha a duração de cada período, com um mínimo para um treino curto continuar legível.
    widths = np.maximum([days["fit"], days["pred"]], 0.35 * sum(days.values()))

    fig = make_subplots(
        rows=3, cols=2, shared_xaxes="columns", shared_yaxes="rows",
        row_heights=[0.44, 0.44, 0.12], column_widths=list(widths / widths.sum()),
        vertical_spacing=0.05, horizontal_spacing=0.02,
        column_titles=[f"<b>{name}</b><br>{period_label(scenario[key])}" for key, name in periods],
    )
    shown = {key: scenario[key][CHANNELS].where(scenario[key]["uptime"]) if hide_stopped else scenario[key][CHANNELS]
             for key, _ in periods}
    limits = {row: axis_limit(pd.concat([shown["fit"][group], shown["pred"][group]]))
              for row, group in enumerate([VELOCITY, ACCELERATION], start=1)}
    capped_legend = False
    for col, (key, _) in enumerate(periods, start=1):
        df, values = scenario[key], shown[key]
        for row, group in enumerate([VELOCITY, ACCELERATION], start=1):
            for column in group:
                axis = column[-1]
                fig.add_trace(go.Scatter(
                    x=values.index, y=values[column], name=f"Eixo {axis.upper()}", legendgroup=axis,
                    showlegend=row == 1 and col == 1, mode="lines", connectgaps=False,
                    line=dict(color=AXIS_COLORS[axis], width=1),
                    hovertemplate="%{y:.3g}<extra>" + f"eixo {axis.upper()}</extra>",
                ), row=row, col=col)
                # Leituras acima do limite do eixo: um marcador na borda superior, valor real no hover.
                above = values[column][values[column] > limits[row]]
                if len(above):
                    fig.add_trace(go.Scatter(
                        x=above.index, y=np.full(len(above), limits[row] * 0.93), customdata=above.values,
                        mode="markers", name="Acima do limite do eixo", legendgroup="capped",
                        showlegend=not capped_legend, marker=dict(symbol="triangle-up", size=9,
                        color=AXIS_COLORS[axis], line=dict(color="#52514e", width=1)),
                        hovertemplate="%{customdata:.3g}<extra>" + f"eixo {axis.upper()} (fora da escala)</extra>",
                    ), row=row, col=col)
                    capped_legend = True
        fig.add_trace(go.Scatter(
            x=df.index, y=df["uptime"].astype(int), mode="lines", showlegend=False, hoverinfo="skip",
            line=dict(color=NEUTRAL, width=1, shape="hv"), fill="tozeroy", fillcolor="rgba(138, 137, 133, 0.35)",
        ), row=3, col=col)

    for incident in valid_incidents(scenario_id):
        fig.add_vrect(x0=incident.start, x1=incident.end, fillcolor=INCIDENT_FILL, line_width=0,
                      layer="below", row="all", col=2)
    if valid_incidents(scenario_id):
        # Entrada da legenda para as faixas de incidente.
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Incidente rotulado",
                                 marker=dict(color=INCIDENT_FILL, symbol="square", size=14)), row=1, col=2)

    n_valid = len(valid_incidents(scenario_id))
    n_invalid = len(scenario["incidents"]) - n_valid
    gap = (scenario["pred"].index[0] - scenario["fit"].index[-1]).total_seconds() / 86400
    running = pd.concat([scenario["fit"]["uptime"], scenario["pred"]["uptime"]]).mean()
    subtitle = (f"{n_valid} incidente(s) rotulado(s)"
                + (f" + {n_invalid} com fim antes do início (não desenhado)" if n_invalid else "")
                + f" · treino {days['fit']:.0f} d, intervalo {gap:.1f} d, predição {days['pred']:.0f} d"
                + f" · rodando {running:.0%} do tempo")
    style(fig, f"Cenário {scenario_id}", height=640, subtitle=subtitle)
    fig.update_layout(margin_t=fig.layout.margin.t + 16)  # espaço para os títulos de coluna em duas linhas
    fig.update_annotations(font_size=13)
    fig.update_yaxes(range=[0, limits[1]], row=1)
    fig.update_yaxes(range=[0, limits[2]], row=2)
    fig.update_yaxes(title_text="RMS da velocidade<br>(mm/s)", row=1, col=1)
    fig.update_yaxes(title_text="RMS da<br>aceleração (g)", row=2, col=1)
    fig.update_yaxes(showticklabels=False, showgrid=False, range=[0, 1.1], row=3)
    fig.update_yaxes(title_text="Rodando", row=3, col=1)
    fig.update_xaxes(hoverformat="%d/%m/%Y %H:%M")
    fig.update_xaxes(nticks=6, tickformat="%d/%m/%y", row=3)
    fig.show()


Todos os cenários, em ordem. Qualquer um pode ser redesenhado com `plot_scenario(scenario_id, hide_stopped=False)`.


In [ ]:
for scenario_id in scenarios:
    plot_scenario(scenario_id)


## Conclusões

**Os dados estão limpos; a dificuldade está em outro lugar.** Não há leituras ausentes, negativas ou duplicadas. O que varia é a quantidade de histórico (de 4 a 31 dias de treino, com apenas 291 leituras rodando no menor caso), a taxa de amostragem (10 min em 26 cenários, 4 min no cenário 3, 1 min no cenário 11) e a continuidade (buracos de até 10 h no cenário 28). Por isso as janelas precisam ser definidas em tempo, comportam de 12 a 120 leituras e podem ficar quase vazias. Esses buracos também encurtam a predição: o janelamento do pipeline para na primeira janela sem leituras, então com janelas de 2 h os cenários 5, 27 e 28 perdem as últimas 120 a 266 horas (medido e contornado em `detection_analysis.ipynb`, seção 1).

**O `uptime` é confiável, e as leituras paradas precisam sair.** Com a máquina parada, a aceleração fica no piso de ruído do sensor (0,01 a 0,03 g) em todos os cenários, de 4 a 700 vezes abaixo do nível de operação. Menos de 0,3% das leituras paradas parecem leituras rodando, e uma mediana de 0,2% das leituras rodando fica no piso de ruído (22% no cenário 16, cujo nível rodando é só 4 vezes o piso). As máquinas ficam paradas entre 0% e 74% do tempo, então manter essas leituras puxaria o baseline para zero e inflaria sua dispersão. O modelo descarta toda leitura com `uptime == False`, no treino e na predição.

**A velocidade é um canal fraco em máquinas silenciosas.** O RMS da velocidade tem um piso, com a máquina parada, de 0,2 a 1,6 mm/s. Nos cenários 7, 8, 14, 16, 23 e 26 a velocidade rodando é menor que 3 vezes esse piso, então a maior parte da informação ali está na aceleração.

**As partidas leem baixo, não alto.** 30% das primeiras leituras depois de uma parada ficam abaixo de metade do nível rodando e só 4% acima do dobro. Um detector que procura vibração *alta* não é perturbado por elas, então não recebem tratamento especial.

**O logaritmo não deixa os canais simétricos.** As amplitudes brutas já são quase simétricas (assimetria mediana de 0,13); o logaritmo estica as leituras baixas e deixa uma cauda *inferior* longa (assimetria mediana de -1,26; só 37% das séries ficam mais próximas da simetria). O motivo para trabalhar com a amplitude em log é outro: uma falha multiplica a amplitude, e um desvio relativo significa o mesmo num ativo rodando a 0,05 g e num rodando a 8 g. Duas consequências para o modelo: usar estatísticas robustas (mediana e MAD) no baseline e julgar só desvios na direção de vibração mais alta.

**Os seis canais se movem juntos.** A correlação mediana entre pares é 0,83. Um modelo conjunto consegue resumir o comportamento normal em poucas direções e distinguir "tudo subiu" de "um eixo subiu sozinho", que é o que o PCA com T² e SPE faz. A correlação é fraca em alguns ativos (percentil 5 de 0,05), então o número de componentes precisa ser escolhido por ativo.

**O baseline nem sempre se mantém.** Os períodos de treino e de predição chegam a ficar 62 dias distantes (10 dias ou mais em 9 cenários). Em 8 cenários o nível normal do período de predição fica a mais de 2 sigma do baseline do treino em algum canal; no cenário 10 fica a 24 sigma durante o período inteiro. Um baseline fixo vai gerar ali alertas que os rótulos não reconhecem.

**Os rótulos marcam uma janela, não o período anormal inteiro.**
- 29 incidentes em 22 cenários; 7 cenários não têm nenhum. Um incidente (cenário 7) termina antes de começar e não pode ser encontrado, então o recall fica limitado a 28/29.
- Quatro cenários têm incidentes com menos de 4 h (2, 3, 11, 13): um alerta precisa ser confirmado em duas ou três horas para cair dentro deles.
- Em 10 incidentes algum canal já está acima de 3 sigma nas 24 h *antes* da janela, e em 11 ainda está nas 24 h *depois*. Um alerta gerado quando o desvio realmente começa conta como falso positivo, e só um alerta posterior dentro da janela pontua. Isso faz a lógica de re-alerta importar para a métrica, não só para o produto.
- Em 9 dos 28 incidentes válidos nenhum canal fica acima de 3 sigma em nível mediano durante a janela (cenários 5, 6, 9, 11, 12, 14, 17, 27). Não dá para esperar que um detector baseado no nível de vibração encontre esses incidentes; eles precisariam de features que descrevam outra coisa além do nível (dispersão, surtos, relações entre os eixos).